# Lending Club Loan Data Analysis

## Problem Statement
For companies like Lending Club, correctly predicting whether or not a loan will be a default is very important. In this project, using historical data from 2007 to 2015, we build a deep learning model to predict the chance of default for future loans. This dataset is highly imbalanced and includes several features that make the problem challenging.

**Objective:** Create a model that predicts whether or not a loan will be default using historical data.

**Domain:** Finance

**Tasks:**
1. Feature Transformation
2. Exploratory Data Analysis (EDA)
3. Additional Feature Engineering (Correlation analysis)
4. Modeling (Keras/TensorFlow)

### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

print(f"TensorFlow version: {tf.__version__}")

### Load Dataset

In [ ]:
# Load data - handling \r line endings if necessary
df = pd.read_csv('loan_data.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

### Task 1: Feature Transformation
Transform categorical values into numerical values.

In [ ]:
print(f"Categorical columns: {df.select_dtypes(include=['object']).columns.tolist()}")

# Transform 'purpose' column using dummy variables
df_transformed = pd.get_dummies(df, columns=['purpose'], drop_first=True)

print(f"New shape after transformation: {df_transformed.shape}")
df_transformed.head()

### Task 2: Exploratory Data Analysis (EDA)
Explore different factors in the dataset.

In [ ]:
# 1. Target Variable Distribution
plt.figure(figsize=(8, 5))
sns.countplot(x='not.fully.paid', data=df, palette='viridis')
plt.title('Distribution of Loan Status (0 = Paid, 1 = Not Fully Paid)')
plt.show()

print(f"Target distribution:\n{df['not.fully.paid'].value_counts(normalize=True) * 100}")

In [ ]:
# 2. FICO Score Distribution by Loan Status
plt.figure(figsize=(10, 6))
df[df['not.fully.paid'] == 0]['fico'].hist(alpha=0.5, color='blue', bins=30, label='not.fully.paid=0')
df[df['not.fully.paid'] == 1]['fico'].hist(alpha=0.5, color='red', bins=30, label='not.fully.paid=1')
plt.legend()
plt.xlabel('FICO Score')
plt.title('FICO Distribution by Loan Status')
plt.show()

In [ ]:
# 3. Purpose of Loan vs Loan Status
plt.figure(figsize=(12, 6))
sns.countplot(x='purpose', hue='not.fully.paid', data=df, palette='Set1')
plt.xticks(rotation=45)
plt.title('Loan Purpose vs Status')
plt.show()

### Task 3: Additional Feature Engineering
Check correlation between features and drop strongly correlated features.

In [ ]:
# Calculate correlation matrix
plt.figure(figsize=(14, 10))
sns.heatmap(df_transformed.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.show()

In [ ]:
# Function to identify highly correlated features
def get_high_corr_pairs(df, threshold=0.8):
    corr_matrix = df.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
    return to_drop

cols_to_drop = get_high_corr_pairs(df_transformed, 0.8)
print(f"Features to drop (> 0.8 correlation): {cols_to_drop}")

df_final = df_transformed.drop(columns=cols_to_drop)
print(f"Final features: {df_final.columns.tolist()}")

### Task 4: Modeling
Build a deep learning model using Keras.

In [ ]:
# Prepare data
X = df_final.drop('not.fully.paid', axis=1)
y = df_final['not.fully.paid']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# Build Model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    BatchNormalization(),
    Dropout(0.2),
    
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    
    Dense(16, activation='relu'),
    
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.summary()

In [ ]:
# Train Model
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_test_scaled, y_test),
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Evaluate Model
y_pred_prob = model.predict(X_test_scaled)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

print("Classification Report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

print(f"AUC-ROC Score: {roc_auc_score(y_test, y_pred_prob):.4f}")

# Conclusion
In this project, we successfully implemented a deep learning pipeline for Lending Club loan default prediction. 
We performed feature transformation on categorical variables, explored the data visually, reduced dimensionality through correlation analysis, and built a Keras-based neural network. The model provides a foundation for assessing loan risk.